# 16 — Fight dynamics vs. Vegas (method props)

**Goal:** Compare **machine-learned** distributions over *how the fight ends* (six sportsbook buckets) to **de-vigged** prices from the Kaggle odds file (`f1_ko_odds`, `f2_ko_odds`, `f1_sub_odds`, `f2_sub_odds`, `f1_dec_odds`, `f2_dec_odds`).

**Label (6-class):** Which bucket occurred — fighter_1 KO/TKO, fighter_2 KO/TKO, fighter_1 sub, fighter_2 sub, fighter_1 decision, fighter_2 decision — mapped from `Method` + `Win_A` on the **first** row per `Fight_Id` (aligned with `fighter_1` / `fighter_2` in the odds table after **name matching**).

**Models:** `HistGradientBoostingClassifier` (multiclass) on **Z-score deltas** `A − B`, same **event-ordered 80/20 split** as notebooks 13/15.

**Also:** Binary **finish vs. decision** (pool KO+SUB vs DEC) with Brier score vs. Vegas implied `P(finish)`.

**Limitation:** Career Z-profiles are not strictly pre-fight; props are sparse for older cards in this scrape. Treat as **descriptive** comparison on the overlap.

**Inputs:** `ufc_fight_stats_cleaned.csv`, `ufc_gmm_comparison.csv`, `data/raw/kaggle_odds/UFC_betting_odds.csv`

In [1]:
# ========================================================================
# NOTEBOOK_CODE_HEADER v1
# File: 16_fight_dynamics_vegas.ipynb | code cell index: 1
# Section: setup, map UFC Method -> six Vegas buckets
# ========================================================================
# INLINE_WORKFLOW_SUMMARY v1
#   • Build one row per fight; map Method text to f1_ko / f2_ko / … / f2_dec.
#   • Drop NC/DQ/overturned-style rows for a clean 6-way label.

import numpy as np
import pandas as pd

RAW_ODDS = "../data/raw/kaggle_odds/UFC_betting_odds.csv"
FIGHTS = "../data/processed/ufc_fight_stats_cleaned.csv"
STYLES = "../data/processed/ufc_gmm_comparison.csv"

PROP_ODDS = [
    "f1_ko_odds",
    "f2_ko_odds",
    "f1_sub_odds",
    "f2_sub_odds",
    "f1_dec_odds",
    "f2_dec_odds",
]
CLASS_NAMES = ["f1_ko", "f2_ko", "f1_sub", "f2_sub", "f1_dec", "f2_dec"]


def map_method_to_bucket(row):
    """Map UFCstats Method + Win_A (Fighter column won?) to odds-table bucket names."""
    mu = str(row["Method"]).strip().upper()
    win_a = int(row["Win_A"]) == 1
    # Non-outcomes / metadata fights
    bad = ("OVERTURNED", "CNC", " DQ", "DQ", "NO CONTEST", "DECLARED")
    if any(b in mu for b in bad):
        return None
    is_dec = ("DEC" in mu) or mu.startswith("U-") or mu.startswith("S-") or mu.startswith("M-")
    is_sub = mu.startswith("SUB")
    is_ko = ("KO" in mu) or ("TKO" in mu)
    # Decisions: U-DEC, S-DEC, M-DEC, etc.
    if is_dec and not is_ko and not is_sub:
        return "f1_dec" if win_a else "f2_dec"
    if is_sub:
        return "f1_sub" if win_a else "f2_sub"
    if is_ko:
        return "f1_ko" if win_a else "f2_ko"
    return None


def norm_name(x):
    return str(x).strip().lower() if pd.notna(x) else ""


fights = pd.read_csv(FIGHTS)
f1 = fights.drop_duplicates("Fight_Id", keep="first").copy()
f1["Fight_Id"] = f1["Fight_Id"].astype(str)
f1["Win_A"] = f1["Won"].astype(int)
f1["outcome_bucket"] = f1.apply(map_method_to_bucket, axis=1)

styles = pd.read_csv(STYLES)
cluster_col = "Cluster_k5"
z_cols = ["Sig_Str_PM_Z", "Takedown_Att_PM_Z", "Sub_Att_PM_Z", "Control_Ratio_Z"]
z_map = styles[["Fighter"] + z_cols].set_index("Fighter")

base = f1[["Fight_Id", "Event_Id_x", "Fighter", "Opponent", "Win_A", "outcome_bucket"]].rename(
    columns={"Fighter": "Fighter_A", "Opponent": "Fighter_B"}
)
base = base.dropna(subset=["outcome_bucket"]).copy()

uf = base.merge(z_map, left_on="Fighter_A", right_index=True, how="inner")
uf = uf.merge(
    z_map,
    left_on="Fighter_B",
    right_index=True,
    how="inner",
    suffixes=("", "_B"),
)
for c in z_cols:
    uf[f"delta_{c}"] = uf[c] - uf[f"{c}_B"]
feat_cols = [f"delta_{c}" for c in z_cols]

print(f"Fights with styled Z-diffs + mappable 6-way outcome: {len(uf)}")
print(uf["outcome_bucket"].value_counts())

Fights with styled Z-diffs + mappable 6-way outcome: 5471
outcome_bucket
f1_dec    1629
f2_dec    1102
f1_ko     1039
f2_ko      705
f1_sub     650
f2_sub     346
Name: count, dtype: int64


In [2]:
# ========================================================================
# NOTEBOOK_CODE_HEADER v1
# File: 16_fight_dynamics_vegas.ipynb | code cell index: 2
# Section: dedupe US odds rows with all six prop prices; de-vig
# ========================================================================
# INLINE_WORKFLOW_SUMMARY v1
#   • Same dedupe idea as notebook 14: latest adding_date per Fight_Id (US).
#   • Require all six decimal odds > 1 so implied probs are well-defined.

od = pd.read_csv(RAW_ODDS, low_memory=False)
od = od[od["region"].astype(str).str.lower() == "us"].copy()
od["Fight_Id"] = od["fight_url"].astype(str).str.rstrip("/").str.split("/").str[-1]
ad = pd.to_datetime(od["adding_date"], utc=True, format="mixed")
od["adding_date"] = ad.dt.tz_localize(None)

for c in PROP_ODDS:
    od[c] = pd.to_numeric(od[c], errors="coerce")

od = od.dropna(subset=PROP_ODDS)
od = od[(od[PROP_ODDS] > 1.0).all(axis=1)]
idx = od.groupby("Fight_Id")["adding_date"].idxmax()
od1 = od.loc[idx].copy()

inv = 1.0 / od1[PROP_ODDS].values
s = inv.sum(axis=1, keepdims=True)
p_vegas = inv / s
for j, name in enumerate(CLASS_NAMES):
    od1[f"p_{name}"] = p_vegas[:, j]

print(f"US fights with all six method props (deduped): {len(od1)}")

US fights with all six method props (deduped): 4867


In [3]:
# ========================================================================
# NOTEBOOK_CODE_HEADER v1
# File: 16_fight_dynamics_vegas.ipynb | code cell index: 3
# Section: merge odds to fights; name-check fighter_1 == Fighter_A
# ========================================================================

m = uf.merge(
    od1[["Fight_Id", "fighter_1", "fighter_2"] + [f"p_{n}" for n in CLASS_NAMES]],
    on="Fight_Id",
    how="inner",
)
m["nA"] = m["Fighter_A"].map(norm_name)
m["n1"] = m["fighter_1"].map(norm_name)
m["nB"] = m["Fighter_B"].map(norm_name)
m["n2"] = m["fighter_2"].map(norm_name)
# Require consistent corner labeling with the odds file
m["name_ok"] = (m["nA"] == m["n1"]) & (m["nB"] == m["n2"])
m = m.loc[m["name_ok"]].copy()

y_code = m["outcome_bucket"].map({n: i for i, n in enumerate(CLASS_NAMES)})
m["y6"] = y_code
m = m.dropna(subset=["y6"]).copy()
m["y6"] = m["y6"].astype(int)

# Finish = any KO/TKO or SUB bucket (first four classes)
m["y_finish"] = m["y6"].lt(4).astype(int)
p_finish_vegas = m[[f"p_{n}" for n in CLASS_NAMES[:4]]].sum(axis=1)
m["p_finish_vegas"] = p_finish_vegas

print(f"Rows after name alignment: {len(m)}")

sorted_events = np.sort(m["Event_Id_x"].unique())
split_i = max(1, int(0.8 * len(sorted_events)))
train_ev = set(sorted_events[:split_i])
tr = m["Event_Id_x"].isin(train_ev)
te = ~tr

P_COLS = [f"p_{n}" for n in CLASS_NAMES]
Xtr, Xte = m.loc[tr, feat_cols], m.loc[te, feat_cols]
y6_tr, y6_te = m.loc[tr, "y6"], m.loc[te, "y6"]
yf_tr, yf_te = m.loc[tr, "y_finish"], m.loc[te, "y_finish"]

print(f"Train fights: {tr.sum()}  Test fights: {te.sum()}")

Rows after name alignment: 3521
Train fights: 2835  Test fights: 686


In [4]:
# ========================================================================
# NOTEBOOK_CODE_HEADER v1
# File: 16_fight_dynamics_vegas.ipynb | cell 4
# Section: HistGradientBoosting multiclass + finish binary vs Vegas
# ========================================================================

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss

EPS = 1e-6

clf6 = HistGradientBoostingClassifier(
    max_iter=200,
    max_depth=8,
    learning_rate=0.06,
    random_state=42,
)
clf6.fit(Xtr, y6_tr)
p6 = clf6.predict_proba(Xte)
p6 = np.clip(p6, EPS, 1.0 - EPS)
p6 = p6 / p6.sum(axis=1, keepdims=True)

vegas6 = m.loc[te, P_COLS].values
vegas6 = np.clip(vegas6, EPS, 1.0 - EPS)
vegas6 = vegas6 / vegas6.sum(axis=1, keepdims=True)

ll_hgb = log_loss(y6_te, p6)
ll_vegas = log_loss(y6_te, vegas6)
acc_hgb = accuracy_score(y6_te, p6.argmax(axis=1))
acc_vegas = accuracy_score(y6_te, vegas6.argmax(axis=1))

print("=== Six-way method distribution (test events) ===")
print(f"HistGradientBoosting  log loss: {ll_hgb:.4f}   accuracy: {acc_hgb:.3f}")
print(f"Vegas (de-vig 6-way)    log loss: {ll_vegas:.4f}   accuracy: {acc_vegas:.3f}")

clf_f = HistGradientBoostingClassifier(
    max_iter=200,
    max_depth=6,
    learning_rate=0.06,
    random_state=43,
)
clf_f.fit(Xtr, yf_tr)
p_fin = clf_f.predict_proba(Xte)[:, 1]
pv = m.loc[te, "p_finish_vegas"].values

print("\n=== Finish (KO/TKO or SUB) vs decision (test) ===")
print(f"HGB Brier:   {brier_score_loss(yf_te, p_fin):.4f}")
print(f"Vegas Brier: {brier_score_loss(yf_te, pv):.4f}")
print(
    f"HGB acc:   {accuracy_score(yf_te, (p_fin >= 0.5).astype(int)):.3f}"
    f"   Vegas acc: {accuracy_score(yf_te, (pv >= 0.5).astype(int)):.3f}"
)

=== Six-way method distribution (test events) ===
HistGradientBoosting  log loss: 1.9779   accuracy: 0.251
Vegas (de-vig 6-way)    log loss: 1.5567   accuracy: 0.343

=== Finish (KO/TKO or SUB) vs decision (test) ===
HGB Brier:   0.2706
Vegas Brier: 0.2273
HGB acc:   0.506   Vegas acc: 0.624


### Takeaway

- If **Vegas log loss** is lower on the six-way task, the market is better calibrated on *method* for this slice—expected when props are informative.
- **Finish vs. decision** is often easier; compare Brier scores and discuss **coverage** (how many fights have all six prices).
- For the thesis, state clearly that **Z-scores embed future fights** in career rates, so this is not a fair sequential forecasting benchmark—parallel to notebook 13/15.